In [17]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import statsmodels.api as sm


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

## Feature engineering

In [ ]:
def create_features(df):
    df = df.copy()
    df["target"] = df["temperature_2m_mean"].shift(-1)
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
    for w in [3, 7, 14]:
        df[f"roll_mean_{w}"] = df["temperature_2m_mean"].shift(1).rolling(w).mean()
    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    df["day_of_year"] = df.index.dayofyear
    df["month"] = df.index.month
    df["weekday"] = df.index.weekday
    df = df.dropna()
    X = df.drop(columns=["target"])
    y = df["target"]
    return X, y


def create_features_improved(df):
    df = df.copy()
    df['cos_day'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    df['sin_day'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    df['cos_month'] = np.cos(2 * np.pi * df.index.month / 12)
    df['sin_month'] = np.sin(2 * np.pi * df.index.month / 12)
    df['cos_weekday'] = np.cos(2 * np.pi * df.index.weekday / 7)
    df['sin_weekday'] = np.sin(2 * np.pi * df.index.weekday / 7)
    
    df["target"] = df["temperature_2m_mean"].shift(-1)
    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    
    for lag in [1, 7, 365]: 
        if len(df) > lag:
            df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
    # for w in [3, 7, 14]:
    #     df[f"roll_mean_{w}"] = df["temperature_2m_mean"].shift(1).rolling(w).mean()
    df = df.dropna()
    return df.drop(columns=["target"]), df["target"]


def create_features_final(df):
    df = df.copy()
    df["target"] = df["temperature_2m_mean"].shift(-1)
    
    time_periods = {
        "day_of_year": 365.25,
        "month": 12,
        "weekday": 7
    }
    for col, period in time_periods.items():
        if col == "day_of_year": values = df.index.dayofyear
        elif col == "month": values = df.index.month
        else: values = df.index.weekday
        
        df[f"{col}_sin"] = np.sin(2 * np.pi * values / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * values / period)

    for w in [3, 7, 14]:
        df[f"roll_mean_{w}"] = df["temperature_2m_mean"].rolling(w).mean()
        df[f"roll_std_{w}"] = df["temperature_2m_mean"].rolling(w).std()

    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    # df["day_of_year"] = df.index.dayofyear
    # df["month"] = df.index.month
    # df["weekday"] = df.index.weekday
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
        
    df = df.dropna()
    return df.drop(columns=["target"]), df["target"]

## Metrics

In [19]:
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

## XGBoost

In [49]:
def train_xgb(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model = XGBRegressor(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics_ = metrics(y_test, preds)
        print(fold_idx, ", ".join([f"{m}: {s}" for m, s in metrics_.items()]))
        scores.append(metrics_)
    return pd.DataFrame(scores).mean()

## SARIMA

In [42]:
from statsmodels.tsa.deterministic import DeterministicProcess


def train_sarima(series):
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for train_idx, test_idx in tscv.split(series):
        train, test = series.iloc[train_idx], series.iloc[test_idx]
        model = sm.tsa.statespace.SARIMAX(
            train,
            order=(2, 1, 2),
            seasonal_order=(1, 1, 1, 365),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = model.fit(disp=False)
        preds = res.forecast(len(test))
        scores.append(metrics(test, preds))
    return pd.DataFrame(scores).mean()


def train_sarimax(df: pd.DataFrame):
    y = df["temperature_2m_mean"]
    
    # Создаем гармоники Фурье для годовой сезонности (K=5-10 обычно достаточно)
    fourier = DeterministicProcess(
        y.index, constant=True, period=365.25, fourier=5
    ).in_sample()
    
    # exog = df[[
    #     "month",
    #     "day_of_year",
    #     "mean_pressure",
    #     "humidity"
    # ]]
    exog = df.drop(columns=["temperature_2m_mean"])
    
    # Объединяем с вашими exog (давление, влажность)
    exog = pd.concat([fourier, exog], axis=1)
    
    # Обучаем простую модель ARIMAX без внутреннего seasonal_order
    # model = sm.tsa.statespace.SARIMAX(
    #     y, exog=exog, order=(2, 1, 2)
    # )
    
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    
    for train_idx, test_idx in tscv.split(y):
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        X_train, X_test = exog.iloc[train_idx], exog.iloc[test_idx]
        
        model = sm.tsa.statespace.SARIMAX(
            y_train,
            exog=X_train,
            order=(2, 1, 2),
            # seasonal_order=(1, 1, 1, 365)
        )
        
        res = model.fit(disp=False)
        
        preds = res.forecast(len(y_test), exog=X_test)
        
        scores.append(metrics(y_test, preds))
    
    return pd.DataFrame(scores).mean()

## Experiment

In [22]:
def run_benchmark():
    set_seed(42)

    df = pd.read_csv("data_2023_2026.csv", parse_dates=["time"], date_format="%Y-%m-%d")
    df = df.sort_values("time")
    df = df.set_index("time")
    df["temperature_2m_mean"] = df["temperature_2m_mean (°C)"]
    df = df.drop(columns=["temperature_2m_mean (°C)"])

    if "temperature_2m_mean" not in df.columns:
        raise ValueError("No column temperature_2m_mean")

    print(f"Dataset size: {df.shape}")
    X, y = create_features(df)
    print(f"After feature engineering: {X.shape}")

    print("\nTraining XGBoost...")
    xgb_scores = train_xgb(X, y)
    print("XGBoost metrics:")
    print(xgb_scores)

    print("\nTraining SARIMA...")
    sarima_scores = train_sarimax(df)
    print("SARIMA metrics:")
    print(sarima_scores)

    print("\n=== FINAL COMPARISON ===")
    results = pd.DataFrame({"XGBoost": xgb_scores, "SARIMA": sarima_scores})
    print(results)
    return results

In [23]:
results = run_benchmark()

Dataset size: (1096, 12)
After feature engineering: (1065, 25)

Training XGBoost...
XGBoost metrics:
MAE     3.017874
RMSE    4.043682
R2      0.528654
dtype: float64

Training SARIMA...


c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. 

MemoryError: Unable to allocate 769. MiB for an array with shape (734, 734, 187) and data type float64

In [26]:
df = pd.read_csv("data_2023_2026.csv", parse_dates=["time"], date_format="%Y-%m-%d")
df = df.sort_values("time")
df = df.set_index("time")
df["temperature_2m_mean"] = df["temperature_2m_mean (°C)"]
df = df.drop(columns=["temperature_2m_mean (°C)"])

In [27]:
df.columns

Index(['apparent_temperature_max (°C)', 'apparent_temperature_min (°C)',
       'precipitation_sum (mm)', 'rain_sum (mm)', 'snowfall_sum (cm)',
       'wind_speed_10m_max (km/h)', 'wind_gusts_10m_max (km/h)',
       'shortwave_radiation_sum (MJ/m²)', 'sunshine_duration (s)',
       'relative_humidity_2m_mean (%)', 'pressure_msl_mean (hPa)',
       'temperature_2m_mean'],
      dtype='object')

In [78]:
print(f"Dataset size: {df.shape}")
X, y = create_features_final(df)
print(f"After feature engineering: {X.shape}")

Dataset size: (1096, 12)
After feature engineering: (1065, 31)


In [79]:
print("\nTraining XGBoost...")
xgb_scores = train_xgb(X, y)
print("XGBoost metrics:")
print(xgb_scores)


Training XGBoost...
0 MAE: 4.227361854035302, RMSE: 6.372195464903963, R2: 0.4361099134171422
1 MAE: 1.6923959883677084, RMSE: 2.1145042987922578, R2: 0.8695765961605769
2 MAE: 1.9062771639812968, RMSE: 2.708225839558302, R2: 0.6455891353375849
3 MAE: 1.5606269105006074, RMSE: 2.036139544983643, R2: 0.8671837533824776
4 MAE: 2.1411692455600377, RMSE: 2.8635657392611202, R2: 0.8538433829524018
XGBoost metrics:
MAE     2.305566
RMSE    3.218926
R2      0.734461
dtype: float64


In [45]:
print("\nTraining SARIMA...")
sarima_scores = train_sarimax(df)
print("SARIMA metrics:")
print(sarima_scores)


Training SARIMA...


c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Progra

SARIMA metrics:
MAE      29.282346
RMSE     38.351799
R2     -118.258815
dtype: float64
